# Clase 11 · Varias variables a la vez, y cómo saber si el modelo sirve

**Estadística Descriptiva e Inferencial** · Módulo 4 · Sesión 11 de 14

---

## De dónde venimos y a dónde vamos

En la Clase 10 explicamos el tiempo de resolución con **una sola** variable: la
experiencia. Pero el tiempo también depende de **qué tan complicado es el caso**.

Y aquí hay una trampa: si los analistas con más experiencia reciben los casos más
difíciles, mirar solo la experiencia puede llevarte a una conclusión **completamente
equivocada**.

Es exactamente el problema de la paradoja de Simpson (Clase 9), ahora con números.
**Hoy lo resolvemos.**

## Las dos mitades de la clase

| | Pregunta | Herramienta |
|---|---|---|
| **Primera mitad** | ¿Cómo meto varias variables en un modelo? | Regresión múltiple |
| **Segunda mitad** | ¿Cómo sé si mi modelo sirve de verdad? | Train / test |

La segunda mitad es nueva en todo el curso: hasta ahora describíamos e inferíamos.
Hoy además **predecimos**, y predecir obliga a preguntarse si el modelo funcionará con
datos que no ha visto.

## El laboratorio

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 12 | Una variable dice que no hay relación. Dos dicen que sí |
| 2 | 10 | Interpretas los coeficientes correctamente |
| 3 | 10 | Descubres que R² **siempre** sube, aunque agregues basura |
| 4 | 13 | Separas los datos en entrenamiento y prueba |
| 5 | 10 | **Ves un modelo que parece mejor y es mucho peor** |

---
## Celda 0 · Preparación

Hoy usamos dos librerías nuevas, las dos ya instaladas en Colab:

- **`statsmodels`** — para ver los coeficientes con sus p-valores
- **`scikit-learn`** — para separar en entrenamiento/prueba y medir el error

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

# ── Los datos: 60 casos de fraude resueltos ──────────────────────────────
n = 60
experiencia = np.random.default_rng(42).integers(2, 49, n).astype(float)
g = np.random.default_rng(3)
complejidad = np.clip(1 + 0.12 * experiencia + g.normal(0, 1.0, n), 1, 10).round(1)
tiempo = (12 - 0.25 * experiencia + 2.2 * complejidad + g.normal(0, 1.2, n)).round(1)

datos = pd.DataFrame({
    "experiencia": experiencia.astype(int),   # meses del analista
    "complejidad": complejidad,               # puntaje del caso, de 1 a 10
    "tiempo": tiempo,                         # dias que tardo en resolverse
})

print(f"{len(datos)} casos resueltos")
print(datos.head(6).to_string(index=False))

---
# Bloque 1 · Una variable dice una cosa, dos dicen otra  ·  12 min

Empecemos como en la Clase 10: solo con la experiencia.

### Ejercicio 1.1 — La regresión simple

`sm.OLS(y, X).fit()` ajusta la recta. Hay que añadir una columna de unos con
`sm.add_constant(X)` para que calcule el intercepto.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
y = datos["tiempo"]
X_simple = None      # sm.add_constant sobre la columna 'experiencia'

modelo_simple = None # sm.OLS(y, X_simple).fit()

coef_exp_simple = None    # el coeficiente de experiencia
p_exp_simple    = None    # su p-valor
r2_simple       = None

print(f"coef = {coef_exp_simple} | p = {p_exp_simple} | R2 = {r2_simple}")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("coeficiente de experiencia (simple)", coef_exp_simple, -0.0058, tol=1e-3),
     check("p-valor", p_exp_simple, 0.8158, tol=1e-3),
     check("R cuadrado", r2_simple, 0.0009, tol=1e-3),
     check_bool("con este modelo, la experiencia parece irrelevante", p_exp_simple > 0.05)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Añade la complejidad del caso

Ahora metemos **las dos** variables. La sintaxis es idéntica: solo cambia la lista de
columnas.

$$tiempo = a + b_1 \cdot experiencia + b_2 \cdot complejidad$$

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X_multiple = None       # sm.add_constant con LAS DOS columnas

modelo_multiple = None

coef_exp_mult  = None   # coeficiente de experiencia
coef_comp_mult = None   # coeficiente de complejidad
r2_multiple    = None

print(f"exp = {coef_exp_mult} | comp = {coef_comp_mult} | R2 = {r2_multiple}")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("coeficiente de experiencia (múltiple)", coef_exp_mult, -0.2229, tol=1e-3),
     check("coeficiente de complejidad", coef_comp_mult, 2.0689, tol=1e-3),
     check("R cuadrado del modelo múltiple", r2_multiple, 0.7083, tol=1e-3),
     check_bool("ahora la experiencia SÍ es significativa",
                modelo_multiple.pvalues["experiencia"] < 0.001)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

### Ejercicio 1.3 — ¿Por qué pasó esto?

La explicación es la misma de la paradoja de Simpson: hay una variable que influye en el
resultado **y** está relacionada con la que estabas mirando.

Compruébalo: ¿están relacionadas la experiencia y la complejidad?

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
corr_exp_comp = None    # correlación entre experiencia y complejidad

print("correlación:", corr_exp_comp)

In [ ]:
# ── VERIFICACIÓN 1.3 ─────────────────────────────────────────────────────
r = [check("correlación entre experiencia y complejidad", corr_exp_comp, 0.808, tol=1e-2),
     check_bool("están fuertemente relacionadas", corr_exp_comp > 0.7)]
print()
print("Esta es la respuesta al problema que dejo la Clase 9:")
print("  la regresion multiple permite CONTROLAR una variable de confusion,")
print("  metiendola en el modelo.")
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.3")

---
# Bloque 2 · Cómo se leen los coeficientes  ·  10 min

Aquí está la frase que hay que memorizar, porque es lo que distingue la regresión múltiple
de la simple:

> Cada coeficiente dice cuánto cambia `y` cuando esa variable sube una unidad
> **y todas las demás se mantienen constantes**.

Esa última parte —«manteniendo lo demás constante»— es todo el valor de la técnica.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Escribe la ecuación del modelo con sus tres números, y traduce cada
# coeficiente a una frase de negocio.

intercepto = None
b_exp      = None
b_comp     = None

print(f"tiempo = {intercepto} + {b_exp} * experiencia + {b_comp} * complejidad")

In [ ]:
# ── VERIFICACIÓN 2 ──────────────────────────────────────────────────────
pred_ejemplo = intercepto + b_exp * 24 + b_comp * 5.0
r = [check("intercepto", intercepto, 11.7783, tol=1e-3),
     check("predicción para 24 meses y complejidad 5", pred_ejemplo, 16.7728, tol=1e-2)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa el bloque 2")

---
# Bloque 3 · R² siempre sube. Siempre.  ·  10 min

Aquí hay una propiedad incómoda de R² que hay que conocer:

> **Cada variable que añades hace subir el R², aunque esa variable sea basura pura.**

Vamos a comprobarlo de la forma más brutal posible: agregando columnas de **números
aleatorios** que no tienen absolutamente nada que ver con el tiempo de resolución.

In [ ]:
# ── DEMOSTRACIÓN: creamos 20 columnas de ruido puro ──────────────────────
rng = np.random.default_rng(99)
datos_ruido = datos.copy()

for k in range(1, 21):
    datos_ruido[f"ruido{k}"] = rng.normal(0, 1, n).round(3)

print("Estas columnas son numeros al azar. No miden nada.")
print(datos_ruido[["tiempo", "ruido1", "ruido2", "ruido3"]].head(4).to_string(index=False))

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
base = ["experiencia", "complejidad"]

filas = []
for k in [0, 5, 10, 15, 20]:
    columnas = base + [f"ruido{i}" for i in range(1, k + 1)]
    m = None            # ajusta el modelo con esas columnas
    filas.append({"variables": len(columnas), "de_ruido": k,
                  "R2": None, "R2_ajustado": None})

tabla_r2 = pd.DataFrame(filas)
print(tabla_r2)

In [ ]:
# ── VERIFICACIÓN 3 ──────────────────────────────────────────────────────
r2_0  = tabla_r2.loc[tabla_r2.de_ruido == 0, "R2"].iloc[0]
r2_20 = tabla_r2.loc[tabla_r2.de_ruido == 20, "R2"].iloc[0]
aj_0  = tabla_r2.loc[tabla_r2.de_ruido == 0, "R2_ajustado"].iloc[0]
aj_20 = tabla_r2.loc[tabla_r2.de_ruido == 20, "R2_ajustado"].iloc[0]

r = [check("R2 con las 2 variables buenas", r2_0, 0.7083, tol=1e-3),
     check("R2 con 20 columnas de ruido", r2_20, 0.7937, tol=1e-3),
     check_bool("el R2 SUBIÓ al agregar basura", r2_20 > r2_0),
     check_bool("pero el R2 ajustado BAJÓ", aj_20 < aj_0)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa el bloque 3")

---
# Bloque 4 · Entrenamiento y prueba  ·  13 min

El R² ajustado ayuda, pero hay una forma mucho mejor —y más honesta— de saber si un
modelo sirve:

> **Guardas una parte de los datos, no dejas que el modelo los vea, y después le pides que
> los prediga.**

Es la idea más importante del análisis predictivo, y es de sentido común: si un modelo solo
funciona con los datos que ya conoce, no sirve para nada.

| | Para qué |
|---|---|
| **Entrenamiento** (70 %) | El modelo aprende de aquí |
| **Prueba** (30 %) | El modelo **nunca los ve** hasta el final |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X = datos[["experiencia", "complejidad"]]
y = datos["tiempo"]

# train_test_split devuelve CUATRO cosas, en este orden:
X_train, X_test, y_train, y_test = None, None, None, None

print(f"entrenamiento: {len(X_train)} casos")
print(f"prueba:        {len(X_test)} casos")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("casos de entrenamiento", len(X_train), 42),
     check("casos de prueba", len(X_test), 18),
     check_bool("juntos suman el total", len(X_train) + len(X_test) == len(X)),
     check_bool("no se solapan",
                len(set(X_train.index) & set(X_test.index)) == 0,
                "un caso no puede estar en los dos grupos")]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — Entrena y evalúa

Ahora entrenamos con `scikit-learn`, que es la librería estándar para esto.

Y medimos con tres números:

| Métrica | Qué es | Se lee |
|---|---|---|
| **R²** | fracción de variación explicada | más alto es mejor |
| **RMSE** | error típico, en las unidades de y | más bajo es mejor |
| **MAE** | error absoluto promedio | más bajo es mejor |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelo = None       # LinearRegression().fit(...) usando SOLO los de entrenamiento

pred_train = None
pred_test  = None

r2_train, r2_test     = None, None
rmse_train, rmse_test = None, None   # raíz de mean_squared_error

print(f"R2 train: {r2_train} | R2 test: {r2_test}")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("R2 en entrenamiento", r2_train, 0.7089, tol=1e-3),
     check("R2 en prueba", r2_test, 0.6722, tol=1e-3),
     check("RMSE en prueba", rmse_test, 1.2817, tol=1e-2),
     check_bool("el rendimiento en prueba es parecido al de entrenamiento",
                abs(r2_train - r2_test) < 0.15,
                "si la diferencia fuera enorme, habria sobreajuste")]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.2")

---
# Bloque 5 · Un modelo que parece mejor y es mucho peor  ·  10 min

**Este es el bloque más importante del día.**

Volvemos a las 20 columnas de ruido del bloque 3. Ahí vimos que hacían subir el R².

Ahora vamos a hacer la pregunta que de verdad importa: **¿mejoran las predicciones en datos
que el modelo no ha visto?**

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Usa los MISMOS casos de entrenamiento y prueba de antes, pero con modelos
# que van incorporando cada vez más columnas de ruido.

idx_train = X_train.index
idx_test  = X_test.index

filas = []
for k in [0, 5, 10, 15, 20]:
    columnas = base + [f"ruido{i}" for i in range(1, k + 1)]
    Xk = datos_ruido[columnas]
    mod = None       # entrena con datos_ruido.loc[idx_train]
    filas.append({"variables": len(columnas),
                  "R2_train": None, "R2_test": None, "RMSE_test": None})

tabla_over = pd.DataFrame(filas)
print(tabla_over)

In [ ]:
# ── VERIFICACIÓN 5 ──────────────────────────────────────────────────────
r2tr_0, r2tr_20 = tabla_over.R2_train.iloc[0], tabla_over.R2_train.iloc[-1]
r2te_0, r2te_20 = tabla_over.R2_test.iloc[0], tabla_over.R2_test.iloc[-1]

r = [check("R2 train con 2 variables", r2tr_0, 0.7089, tol=1e-3),
     check("R2 train con 22 variables", r2tr_20, 0.8547, tol=1e-3),
     check("R2 test con 2 variables", r2te_0, 0.6722, tol=1e-3),
     check("R2 test con 22 variables", r2te_20, 0.2067, tol=1e-3),
     check_bool("el R2 de entrenamiento SUBIÓ", r2tr_20 > r2tr_0),
     check_bool("pero el de prueba SE DERRUMBÓ", r2te_20 < r2te_0 - 0.3)]
print()
print("LA LECCION DEL DIA:")
print("  el rendimiento en los datos de entrenamiento NO dice si un modelo sirve.")
print("  Solo el rendimiento en datos que nunca vio lo dice.")
print()
print("LABORATORIO COMPLETO" if all(r) else "Revisa el bloque 5")

---
# Cierre

### Las dos ideas de hoy

**1 · Controlar variables.** Meter una variable en el modelo permite ver el efecto de otra
«manteniendo esa constante». Es la solución al problema que dejó la paradoja de Simpson en
la Clase 9.

**2 · Validar.** Un modelo solo demuestra que sirve prediciendo datos que nunca vio.
El rendimiento en entrenamiento no cuenta.

### Checklist de salida

- [ ] Sé leer un coeficiente diciendo «manteniendo lo demás constante».
- [ ] Sé que una variable omitida puede invertir una conclusión.
- [ ] Sé que R² siempre sube al agregar variables, aunque sean basura.
- [ ] Sé separar los datos en entrenamiento y prueba.
- [ ] Sé reconocer sobreajuste: bien en entrenamiento, mal en prueba.
- [ ] Reporto el error en unidades del negocio (RMSE en días, en soles).

### Los números del día

| | |
|---|---|
| Coef. experiencia, modelo simple | **−0.006** (p = 0.82) |
| Coef. experiencia, con complejidad | **−0.223** (p < 0.001) |
| Correlación experiencia-complejidad | 0.808 |
| R² con 2 variables → con 22 de ruido | 0.708 → **0.794** |
| R² ajustado, lo mismo | 0.698 → **0.671** |
| **R² en prueba, lo mismo** | 0.672 → **0.207** |

### Los tres errores que evita esta clase

**1 · Omitir una variable importante.** El coeficiente de la que sí miras queda
contaminado. Es el error de la Clase 9 en versión numérica.

**2 · Comparar modelos por R².** Siempre gana el que tiene más variables. Usa R² ajustado
o, mejor, prueba en datos nuevos.

**3 · Reportar el rendimiento en entrenamiento.** Es como calificar un examen con las
respuestas a la vista.

### Reto para la próxima clase

Toma un modelo o un análisis de tu trabajo donde se explique un número con otro.

Pregúntate: **¿qué variable falta?** ¿hay algo que influya en el resultado y que además
esté relacionado con la variable que estás usando?

Si la encuentras, métela en el modelo y mira si el coeficiente original cambia.

### Clase 12

Hoy predijimos un **número** (días). En la próxima predecimos un **sí o un no**: ¿este
cliente entrará en mora? ¿esta transacción es fraude?

Eso es la **regresión logística**, y es el modelo que está detrás de casi todo scorecard de
crédito. La validación cambia también: ya no sirve el RMSE, y entran la matriz de confusión
y el AUC.

---
*Estadística Descriptiva e Inferencial · Módulo 4 · Clase 11*